# Heavy Supplier, Inventory & Warehouse Analytics

## Task A1 — Data Profiling & Structure Analysis

### Objective
Understand the structure, size, data types, missing values,
duplicates, and relationships across all available datasets
before performing data cleaning and analysis.

In [1]:
#import libraries
import pandas as pd #working woth datasets
import numpy as np #numerical operations
import os #working with folders/files
import glob #finding multiple csv files

In [27]:
os.listdir('/content')

['.config',
 '.ipynb_checkpoints',
 'heavy_supplier_data',
 'HeavySuppliersWarehouseDatasets.zip',
 'sample_data']

In [29]:
import zipfile
zip_path='/content/HeavySuppliersWarehouseDatasets.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall('/content/heavy_supplier_data')

In [30]:
os.listdir('/content/heavy_supplier_data/HeavySuppliersWarehouseDatasets')

['sales_orders_header.csv',
 'sales_orders_lines.csv',
 'suppliers.csv',
 'purchase_orders_header.csv',
 'customers.csv',
 'notebook',
 'outputs',
 'products.csv',
 'invoices.csv',
 'branches.csv',
 'stock_ledger.csv',
 'purchase_orders_lines.csv',
 'data',
 '.DS_Store',
 'inventory_master.csv',
 'payments.csv']

In [31]:
DATA_PATH='/content/heavy_supplier_data/HeavySuppliersWarehouseDatasets/'

In [32]:
os.listdir(DATA_PATH)

['sales_orders_header.csv',
 'sales_orders_lines.csv',
 'suppliers.csv',
 'purchase_orders_header.csv',
 'customers.csv',
 'notebook',
 'outputs',
 'products.csv',
 'invoices.csv',
 'branches.csv',
 'stock_ledger.csv',
 'purchase_orders_lines.csv',
 'data',
 '.DS_Store',
 'inventory_master.csv',
 'payments.csv']

In [33]:
files=glob.glob(os.path.join(DATA_PATH, '*.csv'))
print('csv files found:', len(files))

for file in files:
  print(os.path.basename(file))

csv files found: 12
sales_orders_header.csv
sales_orders_lines.csv
suppliers.csv
purchase_orders_header.csv
customers.csv
products.csv
invoices.csv
branches.csv
stock_ledger.csv
purchase_orders_lines.csv
inventory_master.csv
payments.csv


In [34]:
#load datasets

datasets={}

for file in files:
  name=os.path.basename(file).replace('.csv', '')
  datasets[name]=pd.read_csv(file)

print('Datasets Loaded:', len(datasets))


Datasets Loaded: 12


In [35]:
#create the datasets summary

dataset_summary=[]

for name, df in datasets.items():
  dataset_summary.append({
      'Datasets': name,
      'Rows': df.shape[0],
      'Columns': df.shape[1],
      'Missing_Values': df.isnull().sum().sum(),
      'Duplicate_Rows': df.duplicated().sum()
  })
dataset_summary=pd.DataFrame(dataset_summary)
dataset_summary

,Datasets,Rows,Columns,Missing_Values,Duplicate_Rows
0,sales_orders_header,20000,11,0,0
1,sales_orders_lines,130402,9,0,0
2,suppliers,8,12,0,0
3,purchase_orders_header,24000,10,2370,0
4,customers,500,15,0,0
5,products,30,23,0,0
6,invoices,18033,10,0,0
7,branches,6,13,0,0
8,stock_ledger,237230,9,0,0
9,purchase_orders_lines,155495,9,0,0


In [36]:
#find the structure of every dataset
for name, df in datasets.items():
  print('\n' + '=' * 60)
  print(f'{name}')
  print('=' * 60)
  print('Shape:', df.shape)
  print('Columns')
  print(df.columns.tolist())


sales_orders_header
Shape: (20000, 11)
Columns
['so_id', 'customer_id', 'branch_id', 'order_date', 'delivery_date', 'order_status', 'payment_terms', 'total_order_value', 'total_gst_amount', 'grand_total', 'sales_channel']

sales_orders_lines
Shape: (130402, 9)
Columns
['so_id', 'line_number', 'product_id', 'quantity', 'unit_price', 'gst_rate', 'line_total', 'gst_amount', 'line_grand_total']

suppliers
Shape: (8, 12)
Columns
['supplier_id', 'supplier_name', 'supplier_type', 'product_category', 'city', 'province', 'region', 'pincode', 'lead_time_days', 'reliability_score', 'import_duty_rate', 'china_tax_id']

purchase_orders_header
Shape: (24000, 10)
Columns
['po_id', 'supplier_id', 'branch_id', 'order_date', 'expected_delivery_date', 'received_date', 'po_status', 'total_cost', 'total_gst_amount', 'grand_total']

customers
Shape: (500, 15)
Columns
['customer_id', 'customer_type', 'industry_segment', 'city', 'state', 'pincode', 'region', 'branch_id', 'credit_limit', 'current_balance', 'p

In [37]:
#check datatypes
for name, df in datasets.items():
  print('\n' + '=' * 60)
  print(f'{name}')
  print('=' * 60)
  print(df.dtypes)


sales_orders_header
so_id                 object
customer_id           object
branch_id             object
order_date            object
delivery_date         object
order_status          object
payment_terms         object
total_order_value      int64
total_gst_amount     float64
grand_total          float64
sales_channel         object
dtype: object

sales_orders_lines
so_id                object
line_number           int64
product_id           object
quantity              int64
unit_price            int64
gst_rate              int64
line_total            int64
gst_amount          float64
line_grand_total    float64
dtype: object

suppliers
supplier_id          object
supplier_name        object
supplier_type        object
product_category     object
city                 object
province             object
region               object
pincode               int64
lead_time_days        int64
reliability_score     int64
import_duty_rate      int64
china_tax_id         object
dtype: object

In [38]:
#check missing values
missing_summary=[]

for name, df in datasets.items():
  for column in df.columns:
    missing=df[column].isnull().sum()
    if missing>0:
      missing_summary.append({
          'Datasets': name,
          'Column': column,
          'Missing_Values': missing,
          'Missing %': round(missing / len(df) *100, 2 )
      })
missing_summary=pd.DataFrame(missing_summary)
missing_summary

,Datasets,Column,Missing_Values,Missing %
0,purchase_orders_header,received_date,2370,9.88


In [39]:
#duplicate summary

duplicate_summary=[]

for name, df in datasets.items():
  duplicate_summary.append({
      'Dataset': name,
      'Duplicated Rows': df.duplicated().sum()
  })
duplicate_summary=pd.DataFrame(duplicate_summary)
duplicate_summary

,Dataset,Duplicated Rows
0,sales_orders_header,0
1,sales_orders_lines,0
2,suppliers,0
3,purchase_orders_header,0
4,customers,0
5,products,0
6,invoices,0
7,branches,0
8,stock_ledger,0
9,purchase_orders_lines,0


In [40]:
po_header=datasets['purchase_orders_header']
po_header.groupby('po_status')['received_date'].apply(
    lambda x: x.isnull().sum()
)

,received_date
po_status,
Cancelled,2370
Received,0


In [43]:
project_path = "/content/Heavy_Supplier_Inventory_Warehouse_Analytics"

os.makedirs(f"{project_path}/notebooks", exist_ok=True)
os.makedirs(f"{project_path}/data", exist_ok=True)
os.makedirs(f"{project_path}/outputs", exist_ok=True)

print("Project folders created!")

Project folders created!


In [47]:
!find /content/Heavy_Supplier_Inventory_Warehouse_Analytics maxdepth 2 -type d

/content/Heavy_Supplier_Inventory_Warehouse_Analytics
/content/Heavy_Supplier_Inventory_Warehouse_Analytics/outputs
/content/Heavy_Supplier_Inventory_Warehouse_Analytics/notebooks
/content/Heavy_Supplier_Inventory_Warehouse_Analytics/data
find: ‘maxdepth’: No such file or directory
find: ‘2’: No such file or directory


In [48]:
for file in os.listdir('/content'):
  if file.endswith('.ipynb'):
    print(file)